# SecureLog Engine — Rapport synthétique

**Projet ING3 — Python 3.10+**  
**Membres du trinôme : Lukas MAUFFRÉ, Sacha BEHLOULI-DROLLÉE, Jeanne TONDA**

Ce projet analyse les logs fictifs d'une salle informatique. Une chaîne de
fonctions lit chaque événement, le valide, le normalise et met à jour les
statistiques. Trois règles détectent les blocs d'échecs d'authentification,
les IP blacklistées et les événements critiques.

La démonstration donne **22 événements valides**, **3 lignes rejetées** et
**2 détections de brute force**. Le code métier reste dans les modules `.py` ;
ce Notebook contient uniquement la démonstration et l'explication des résultats.
Les sorties des cellules sont déjà enregistrées.

## Architecture et choix

`main.py → ingestion → parser → processing → analyzer + rules → résultats + audit`

- `ingestion.py` produit des tuples de provenance avec `yield`, puis délègue
  la lecture de plusieurs fichiers à `yield from`.
- `parser.py` crée un dictionnaire typé : date `datetime`, chaînes, payload
  dict/texte/tuple, et métadonnées immuables.
- `processing.py` combine `filter`/`lambda`, `map` et `match-case`.
- `analyzer.py` conserve des compteurs `defaultdict` et des ensembles d'IP.
- `rules.py` reçoit les fonctions comme objets via `*args` et les paramètres
  via `**kwargs`. Une fenêtre bornée de booléens par identité permet de détecter
  les échecs consécutifs avec `while`, `for`, `continue`, `break` et `for...else`.
- `audit.py` inspecte les portées reçues du pipeline ainsi que ses propres
  `globals()` et `locals()`, et exporte seulement noms, types et tailles.

La seule classe métier, `Analyzer`, regroupe l'état des statistiques. Les autres
étapes sont de petites fonctions. Aucune bibliothèque tierce n'est nécessaire.

In [1]:
from pathlib import Path
import json
import sys
import tempfile

ROOT = Path.cwd()
if ROOT.name == "report":
    ROOT = ROOT.parent
if not (ROOT / "main.py").is_file():
    raise RuntimeError("Ouvrir le Notebook depuis le projet ou son dossier report.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from main import run

with tempfile.TemporaryDirectory() as directory:
    output = Path(directory)
    summary = run(
        [ROOT / "data/auth.log", ROOT / "data/network.log"],
        ROOT / "data/blacklist.txt", output,
    )
    # Cette petite liste sert uniquement à illustrer les 15 alertes de démo.
    alerts = [json.loads(line) for line in
              (output / "alerts.jsonl").read_text().splitlines()]
    audit = json.loads((output / "audit.json").read_text())

print("Entrées :", summary["input"])
print("Alertes :", summary["alerts"])

Entrées : {'read': 28, 'ignored': 3, 'invalid': 3, 'valid': 22}
Alertes : {'BLACKLIST': 12, 'BRUTE_FORCE': 2, 'CRITICAL': 1}


## Résultats par niveau

Les trois rejets sont intentionnels : mauvais nombre de champs, IP invalide et
JSON tronqué. Ils permettent de montrer la gestion d'exceptions. Les commentaires
et les lignes vides sont ignorés séparément. Une erreur de sévérité ERROR ne
signifie pas nécessairement une attaque : elle contribue aux statistiques.

In [2]:
for level, count in summary["by_level"].items():
    print(f"{level:8} {'█' * count} {count}")
print("IP actives :", len(summary["active_ips"]))
print("IP ayant généré ERROR ou CRITICAL :", summary["error_ips"])

CRITICAL █ 1
ERROR    ███████████ 11
INFO     █████ 5
WARNING  █████ 5
IP actives : 6
IP ayant généré ERROR ou CRITICAL : ['192.0.2.30', '198.51.100.24', '203.0.113.66']


## Ensembles et détections

La blacklist est un `frozenset` : les règles ne peuvent pas la modifier.
L'intersection désigne les IP actives blacklistées ; la différence les IP
actives non blacklistées. L'union rassemble les deux ensembles et la différence
symétrique contient les IP présentes dans un seul des deux ensembles.

In [3]:
for operation, addresses in summary["sets"].items():
    print(f"{operation} : {addresses}")
print("\nBrute force :")
for alert in alerts:
    if alert["rule"] == "BRUTE_FORCE":
        print(f"{alert['ip']} / {alert['user']} : {alert['message']}")

union : ['192.0.2.10', '192.0.2.20', '192.0.2.250', '192.0.2.30', '192.0.2.40', '198.51.100.24', '203.0.113.66']
intersection : ['198.51.100.24', '203.0.113.66']
difference : ['192.0.2.10', '192.0.2.20', '192.0.2.30', '192.0.2.40']
symmetric_difference : ['192.0.2.10', '192.0.2.20', '192.0.2.250', '192.0.2.30', '192.0.2.40']

Brute force :
203.0.113.66 / admin : 5 échecs consécutifs
198.51.100.24 / root : 5 échecs consécutifs


Les cinq échecs d'admin et de root déclenchent chacun une alerte. Bob a deux
échecs, un succès, puis deux échecs : aucune série de cinq n'existe. Les identités
`(IP, utilisateur)` sont séparées ; les événements réseau n'interrompent pas
les essais d'authentification. Chaque bloc disjoint de cinq échecs produit une
alerte : dix échecs produisent deux alertes.

Les fichiers sont lus dans l'ordre fourni, sans tri global. Les lignes rejetées
ne peuvent pas participer à la détection. Les alertes désignent des suspicions,
à confirmer en examinant les événements et leur provenance.

In [4]:
for name in ("blacklist", "counters", "histories", "analyzer"):
    print(name, audit["pipeline_locals"][name])
print("Module audité :", audit["audit_globals"]["AUDIT_VERSION"])

blacklist {'type': 'frozenset', 'size': 3}
counters {'type': 'dict', 'size': 4}
histories {'type': 'dict', 'size': 1}
analyzer {'type': 'Analyzer'}
Module audité : {'type': 'str', 'size': 3}


## Validation et consommation mémoire

Les **22 tests automatisés** réussissent : parsing texte et JSON, connexions,
champs invalides, traitement paresseux, erreurs strictes, blacklist, seuils,
succès interrompant une série, identités distinctes, événements réseau,
règle personnalisée, ensembles, audit, CLI et conservation des résultats en
cas d'erreur. Commande : `python -m unittest discover -s tests -v`.

Le code a été exécuté sous Python 3.14.7 et sa syntaxe vérifiée en mode Python
3.10. Le contrôle PEP8 ne signale aucune erreur.

Un essai supplémentaire génère un fichier avec une seule IP et un seul utilisateur,
avec neuf échecs puis un succès. Le pipeline complet est exécuté, exports compris.
`tracemalloc` mesure les allocations Python pendant l'analyse ; la génération du
fichier se fait avant la mesure. Ce n'est pas une mesure de toute la mémoire du
processus (RSS), ni un benchmark universel.

| Lignes | Taille du fichier | Pic des allocations Python | Temps observé |
|---:|---:|---:|---:|
| 10,000 | 601,000 octets | 308,007 octets | 0.227 s |
| 100,000 | 6,010,000 octets | 301,007 octets | 1.803 s |

Le volume d'entrée multiplié par dix n'a pas multiplié le pic de mémoire :
les fichiers et alertes ne sont pas accumulés. Une faible variation de mesure
est normale. Les valeurs dépendent du matériel et de l'interpréteur.

L'état croît néanmoins avec les IP, utilisateurs et couples distincts. Les
historiques sont limités à X booléens par couple ; les ensembles exacts restent
proportionnels au nombre d'identités distinctes. Une ligne très volumineuse peut
également consommer beaucoup de mémoire.

## Limites et conclusion

Le programme traite des fichiers finis et émet ses alertes progressivement.
Il ne suit pas automatiquement les nouvelles lignes ajoutées après lecture,
ne bloque aucune IP et ne fusionne pas chronologiquement plusieurs fichiers.
Il conserve les comptes et ensembles nécessaires à une analyse exacte.

L'approche couvre les notions évaluées avec six modules faciles à expliquer.
Le tableau de correspondance détaillé est dans le README. Une évolution possible
serait une fenêtre temporelle d'échecs, au prix de règles d'ordre et d'expiration
supplémentaires.

## Source du cahier des charges

ÉNONCÉ DU PROJET FINAL : ENGINE DE TRAITEMENT ET D'ANALYSE DE FLUX DE DONNÉES
SÉCURISÉ EN TEMPS RÉEL, niveau ING3. Pièce jointe DOCX de la conversation fournie.
Le chemin `/mnt/data` appartient à la conversation d'origine ; la pièce jointe
a été récupérée localement pour cette réalisation.

SHA-256 du document utilisé : `ff7233cfbe4798b6f0430fb36807e8e5846657fdf3052d45e39f7ca7de62a23f`.